In [2]:
import copy

import numpy as np
import torch
from sklearn.metrics import roc_auc_score

from avatar.metrics.base import BaseMetric

## Базовый класс для создания своей метрики
Базовый класс содержит в себе три обязательных метода:
* **update** - добавление нового батча на каждой итерации обучения. Входными параметрами является `inputs` - batch на текущей итерации, `outputs` - выход модели, имеет фиксированную структуру, смотреть `**avatar.outputs**`
* **compute** - рассчет метрики
* **reset** - очистка атрибутов класса, вызывается при переходе на новую эпоху
  
```python
# avatar.metircs.base import BaseMetric

class BaseMetric(abc.ABC):
    @abc.abstractmethod
    def update(self, inputs, outputs) -> None:
        raise ValueError("Method must be implemented by child classes")

    @abc.abstractmethod
    def compute(self) -> dict[str, float]:
        raise ValueError("Method must be implemented by child classes")

    @abc.abstractmethod
    def reset(self) -> None:
        raise ValueError("Method must be implemented by child classes")
```


## Пример реализации метрики для рассчета RocAucScore

In [3]:
class RocAucScore(BaseMetric):
    def __init__(self):
        self.preds = []

    def update(self, inputs, outputs):
        targets = inputs["targets"].detach().contiguous().cpu().tolist()
        if outputs.logits.dim() == 1:
            predicted = (
                torch.nn.functional.sigmoid(outputs.logits)
                .detach()
                .contiguous()
                .cpu()
                .tolist()
            )
        else:
            predicted = (
                torch.nn.functional.softmax(outputs.logits, dim=-1)[:, 1]
                .detach()
                .contiguous()
                .cpu()
                .tolist()
            )
        self.preds.append({"targets": targets, "probability": predicted})

    def compute(self):
        """return Dict(metric_name: value)"""
        predicted = []
        targets = []
        for item in self.preds:
            predicted.extend(item["probability"])
            targets.extend(item["targets"])

        result = {}
        result["roc_auc_score"] = roc_auc_score(targets, predicted)
        return result

    def reset(self):
        self.preds = []

In [4]:
class RocAucTasksWrapper(BaseMetric):
    """
    calc roc_auc score for each task
    Also calculate average of all task's roc_auc score
    Args:
        task_name: list[str] - list of values with particular task names
        task_name_column: str - Specifies the column used to partition the data.
            The RocAuc score will be caluclated separetly for each unique value
            in this column
    """

    def __init__(self, task_names, task_name_column="task_name"):
        super().__init__()
        self.task_name_column = task_name_column
        self.task_names = task_names + ["all"]
        self.taskname2metrics = {name: RocAucScore() for name in self.task_names}

    def update(self, inputs, outputs):
        for task_name in self.task_names:
            if task_name == "all":
                self.taskname2metrics[task_name].update(inputs, outputs)
                continue
            task_ids = np.where(np.array(inputs[self.task_name_column]) == task_name)[0]
            task_ids = torch.tensor(task_ids).to(outputs.logits.device)
            current_inputs = {}
            current_outputs = copy.deepcopy(outputs)
            for key in inputs.keys():
                if isinstance(inputs[key], dict):
                    current_inputs[key] = {}
                    for inner_key in inputs[key].keys():
                        current_inputs[key][inner_key] = inputs[key][inner_key][
                            task_ids
                        ]
                elif isinstance(inputs[key], list):
                    current_inputs[key] = [inputs[key][i] for i in task_ids]
                else:
                    current_inputs[key] = inputs[key][task_ids]
            current_outputs.logits = outputs.logits[task_ids]
            self.taskname2metrics[task_name].update(current_inputs, current_outputs)

    def compute_avg_over_task(self, result):
        avg_score = 0.0
        for task_name in self.task_names:
            if task_name != "all":
                avg_score += result[task_name + "_roc_auc_score"]
        avg_score /= len(self.task_names) - 1
        return avg_score

    def compute(self):
        result = {}
        for task_name in self.task_names:
            task_metric_result = self.taskname2metrics[task_name].compute()
            for key in task_metric_result.keys():
                result[task_name + "_" + key] = task_metric_result[key]

        result["avg_roc_auc_score"] = self.compute_avg_over_task(result)
        return result

    def reset(self):
        for task_name, metric in self.taskname2metrics.items():
            metric.reset()
